# Analyze HelpSteer2 All-Method Evaluation Results

This notebook analyzes the full HelpSteer2 all-method merge evaluation results. It compares coefficient choices inside the fixed Rewarded-Soups-style interpolation family using proxy scores, preference-weighted utility, distance to the original preference vector, hyperparameters, and computational cost.

All scores shown here are proxy scores. They are not HelpSteer2 human labels, not reward-model scores, and do not establish global Pareto-front improvement. If a finite-search reference is discussed, use the wording `lambda_best`, meaning the best found setting within the tested candidates.


## 1. Clone or update repository


In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")


## 2. Install lightweight analysis dependencies


In [ ]:
!pip install -q "pandas==2.2.2" "numpy<2.1" tabulate


## 3. Check input files


In [ ]:
from pathlib import Path

input_files = {
    "generations": Path("results/helpsteer2_all_method_generations.csv"),
    "scores": Path("results/helpsteer2_all_method_scores.csv"),
    "summary": Path("results/helpsteer2_all_method_result_summary.csv"),
    "summary_md": Path("results/helpsteer2_all_method_result_summary.md"),
    "summary_json": Path("results/helpsteer2_all_method_result_summary.json"),
    "method_costs": Path("results/helpsteer2_method_costs.csv"),
}

for name, file_path in input_files.items():
    status = "FOUND" if file_path.is_file() else "MISSING"
    print(f"{name:14s} {status:8s} {file_path}")

required = [input_files["generations"], input_files["scores"], input_files["summary"]]
missing_required = [path for path in required if not path.is_file()]
if missing_required:
    raise FileNotFoundError(
        "Run Notebook 17 first. Missing required files: "
        + ", ".join(str(path) for path in missing_required)
    )


## 4. Load result tables


In [ ]:
import pandas as pd
import numpy as np

summary = pd.read_csv(input_files["summary"])
generations = pd.read_csv(input_files["generations"])
scores = pd.read_csv(input_files["scores"])
costs = pd.read_csv(input_files["method_costs"]) if input_files["method_costs"].is_file() else None

print(f"Summary rows: {len(summary)}")
print(f"Generation rows: {len(generations)}")
print(f"Score rows: {len(scores)}")
print(f"Cost rows: {len(costs) if costs is not None else 0}")
display(summary.head())


## 5. Winner per preference

This table selects the highest `mean_utility` setting for each preference vector.


In [ ]:
winner_per_preference = (
    summary.sort_values("mean_utility", ascending=False)
    .groupby("preference_name", as_index=False)
    .first()
)

winner_columns = [
    "preference_name",
    "method",
    "method_family",
    "hyperparameter_id",
    "hyperparameters_json",
    "mean_utility",
    "improvement_over_direct_preference",
    "improvement_over_uniform",
    "l1_distance_to_p",
    "l2_distance_to_p",
]
winner_columns = [column for column in winner_columns if column in winner_per_preference.columns]
display(winner_per_preference[winner_columns])


## 6. Best hyperparameter setting per preference and method


In [ ]:
best_setting_by_method = (
    summary.sort_values("mean_utility", ascending=False)
    .groupby(["preference_name", "method"], as_index=False)
    .first()
)

display(best_setting_by_method[[
    "preference_name",
    "method",
    "hyperparameter_id",
    "hyperparameters_json",
    "mean_utility",
    "l1_distance_to_p",
    "l2_distance_to_p",
]].sort_values(["preference_name", "mean_utility"], ascending=[True, False]))


## 7. Mean utility by method


In [ ]:
mean_utility_by_method = (
    summary.groupby("method", as_index=False)["mean_utility"]
    .mean()
    .sort_values("mean_utility", ascending=False)
)
display(mean_utility_by_method)


## 8. Improvements over direct preference and uniform


In [ ]:
improvement_columns = [
    column
    for column in [
        "preference_name",
        "method",
        "hyperparameter_id",
        "mean_utility",
        "improvement_over_direct_preference",
        "improvement_over_uniform",
    ]
    if column in summary.columns
]

display(summary[improvement_columns].sort_values(
    ["preference_name", "mean_utility"],
    ascending=[True, False],
))


## 9. Distance to preference vector p


In [ ]:
distance_table = summary[[
    "preference_name",
    "method",
    "hyperparameter_id",
    "mean_utility",
    "l1_distance_to_p",
    "l2_distance_to_p",
]].sort_values(["preference_name", "l1_distance_to_p", "mean_utility"], ascending=[True, True, False])

display(distance_table)


## 10. Runtime and cost by method


In [ ]:
runtime_source = costs if costs is not None else summary
runtime_numeric = runtime_source.copy()

for column in ["runtime_seconds", "peak_memory_mb", "solver_iterations"]:
    if column in runtime_numeric.columns:
        runtime_numeric[column] = pd.to_numeric(runtime_numeric[column], errors="coerce")

if "solver_success" in runtime_numeric.columns:
    runtime_numeric["solver_success_numeric"] = (
        runtime_numeric["solver_success"]
        .astype(str)
        .str.lower()
        .map({"true": 1.0, "false": 0.0})
    )

runtime_columns = [column for column in [
    "method",
    "runtime_seconds",
    "peak_memory_mb",
    "solver_iterations",
    "solver_success",
] if column in runtime_numeric.columns]

display(runtime_numeric[runtime_columns].head())

agg_spec = {}
if "runtime_seconds" in runtime_numeric.columns:
    agg_spec["mean_runtime_seconds"] = ("runtime_seconds", "mean")
    agg_spec["max_runtime_seconds"] = ("runtime_seconds", "max")
if "peak_memory_mb" in runtime_numeric.columns:
    agg_spec["mean_peak_memory_mb"] = ("peak_memory_mb", "mean")
if "solver_success_numeric" in runtime_numeric.columns:
    agg_spec["solver_success_rate"] = ("solver_success_numeric", "mean")

if agg_spec:
    runtime_by_method = runtime_numeric.groupby("method", as_index=False).agg(**agg_spec)
    display(runtime_by_method.sort_values(list(agg_spec.keys())[0]))
else:
    print("No runtime or cost columns available.")


## 11. Best method per method family


In [ ]:
family_table = (
    summary.sort_values("mean_utility", ascending=False)
    .groupby(["preference_name", "method_family"], as_index=False)
    .first()
)

display(family_table[[
    "preference_name",
    "method_family",
    "method",
    "hyperparameter_id",
    "mean_utility",
    "l1_distance_to_p",
    "l2_distance_to_p",
]].sort_values(["preference_name", "mean_utility"], ascending=[True, False]))


## 12. Utility vs distance to p


In [ ]:
utility_distance_table = summary[[
    "preference_name",
    "method",
    "hyperparameter_id",
    "mean_utility",
    "l1_distance_to_p",
    "l2_distance_to_p",
]].sort_values(["preference_name", "mean_utility"], ascending=[True, False])

display(utility_distance_table)


## 13. Utility improvement vs runtime


In [ ]:
runtime_improvement_columns = [column for column in [
    "preference_name",
    "method",
    "hyperparameter_id",
    "mean_utility",
    "improvement_over_direct_preference",
    "improvement_over_uniform",
    "runtime_seconds",
    "peak_memory_mb",
] if column in summary.columns]

display(summary[runtime_improvement_columns].sort_values(
    ["preference_name", "mean_utility"],
    ascending=[True, False],
))


## 14. Example generated answers with proxy scores


In [ ]:
example_columns = [column for column in [
    "prompt_id",
    "prompt",
    "preference_name",
    "method",
    "hyperparameter_id",
    "generated_response",
    "helpfulness_proxy",
    "correctness_proxy",
    "coherence_proxy",
    "complexity_proxy",
    "verbosity_proxy",
    "utility",
] if column in scores.columns]

examples = (
    scores.sort_values("utility", ascending=False)
    .groupby("method", as_index=False)
    .head(2)
)
display(examples[example_columns].head(20))


## 15. Save analysis report

This writes a concise Markdown report that can be committed with the result files.


In [ ]:
report_path = Path("results/helpsteer2_all_method_analysis_report.md")

def markdown_table(df, columns):
    available_columns = [column for column in columns if column in df.columns]
    if not available_columns:
        return "No matching columns available.\n"
    return df[available_columns].to_markdown(index=False)

winner_report = winner_per_preference[winner_columns].copy()
utility_method_report = mean_utility_by_method.copy()
distance_report = utility_distance_table.groupby("preference_name", as_index=False).head(5).copy()

runtime_report_text = "No runtime or cost columns available."
if "runtime_by_method" in globals():
    runtime_report_text = runtime_by_method.to_markdown(index=False)

lines = [
    "# HelpSteer2 All-Method Analysis Report",
    "",
    "This report summarizes the full HelpSteer2 all-method merge evaluation. The comparison is inside the fixed Rewarded-Soups-style interpolation family.",
    "",
    "The scores are lightweight proxy scores. They are not HelpSteer2 human labels, not reward-model scores, and do not establish global Pareto-front improvement.",
    "",
    "## Main Winners",
    "",
    markdown_table(winner_report, winner_columns),
    "",
    "## Mean Utility by Method",
    "",
    utility_method_report.to_markdown(index=False),
    "",
    "## Comparison to Direct Preference and Uniform",
    "",
    markdown_table(winner_report, [
        "preference_name",
        "method",
        "hyperparameter_id",
        "mean_utility",
        "improvement_over_direct_preference",
        "improvement_over_uniform",
    ]),
    "",
    "## Utility and Distance to p",
    "",
    markdown_table(distance_report, [
        "preference_name",
        "method",
        "hyperparameter_id",
        "mean_utility",
        "l1_distance_to_p",
        "l2_distance_to_p",
    ]),
    "",
    "## Computational Cost Observations",
    "",
    runtime_report_text,
    "",
    "## Limitations",
    "",
    "- Proxy scores are deterministic heuristics, not HelpSteer2 labels.",
    "- Generated responses do not automatically receive human attribute labels.",
    "- The evaluation uses the fixed prompt set and tested hyperparameter grid only.",
    "- No global Pareto-front improvement is claimed.",
    "",
    "## Recommended Next Step",
    "",
    "Inspect proxy utility together with distance to p and prompt-category behavior, then update the main HelpSteer2 prototype report with careful thesis-safe wording.",
]

report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Saved analysis report to {report_path}")


## 16. Preview saved analysis report


In [ ]:
!cat results/helpsteer2_all_method_analysis_report.md


## 17. Git safety check

It is okay to commit small CSV, JSON, Markdown, and plot outputs if useful. Do not commit `adapters/`, zip files, safetensors, bin files, checkpoints, or model weights.


In [ ]:
!git status
